# 4.2 Creating a RunPod Container

Prior to running this file, a RUNPOD API KEY is needed.
An ssh key is also needed. See the runpod-ssh-how-to.docx file in this same directory for instructions and an example on how to do so.

## Configuration

In [2]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

In [3]:
import runpod
print(runpod.get_gpus())

[{'id': 'AMD Instinct MI300X OAM', 'displayName': 'MI300X', 'memoryInGb': 192}, {'id': 'NVIDIA A100 80GB PCIe', 'displayName': 'A100 PCIe', 'memoryInGb': 80}, {'id': 'NVIDIA A100-SXM4-80GB', 'displayName': 'A100 SXM', 'memoryInGb': 80}, {'id': 'NVIDIA A30', 'displayName': 'A30', 'memoryInGb': 24}, {'id': 'NVIDIA A40', 'displayName': 'A40', 'memoryInGb': 48}, {'id': 'NVIDIA B200', 'displayName': 'B200', 'memoryInGb': 180}, {'id': 'NVIDIA GeForce RTX 3070', 'displayName': 'RTX 3070', 'memoryInGb': 8}, {'id': 'NVIDIA GeForce RTX 3080', 'displayName': 'RTX 3080', 'memoryInGb': 10}, {'id': 'NVIDIA GeForce RTX 3080 Ti', 'displayName': 'RTX 3080 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 3090', 'displayName': 'RTX 3090', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 3090 Ti', 'displayName': 'RTX 3090 Ti', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 4070 Ti', 'displayName': 'RTX 4070 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 4080', 'displayName': 'RTX 4080', 'memoryInGb': 16

In [4]:
# Inteligent GPU type selection based on model size using runpod.get_gpus() and the memoryInGb field
# def select_gpu_type(model_name):
#     gpus = runpod.get_gpus()
#     gpu_map = {gpu['displayName']: gpu['id'] for gpu in gpus if gpu['isAvailable']}
    
#     if "70b" in model_name or "gemma-2-70b" in model_name:
#         return gpu_map.get("NVIDIA A100 80GB") or gpu_map.get("NVIDIA A100 40GB")
#     elif "13b" in model_name or "gemma-2-13b" in model_name:
#         return gpu_map.get("NVIDIA RTX A6000")
#     else:
#         return gpu_map.get("NVIDIA GeForce RTX 4090")

In [6]:
POD_NAME       = "lm-eval-pod-test"
IMAGE_NAME     = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
# GPU_TYPE       = "NVIDIA GeForce RTX 4090"
# GPU_TYPE       = "NVIDIA H200"
GPU_TYPE       = "NVIDIA A40"
RESULTS_FILE   = "/workspace/results.json"     # inside pod
LOCAL_RESULTS  = Path("results.json")          # where to store results locally

## Pod Creation

In [7]:
# ───────────────────────────────────────────────
# 1. Create the pod
# ───────────────────────────────────────────────
pod = runpod.create_pod(
    name=POD_NAME,
    image_name=IMAGE_NAME,
    gpu_type_id=GPU_TYPE,
    gpu_count=1,
    container_disk_in_gb=200,
    volume_in_gb=0,
    min_vcpu_count=4,
    min_memory_in_gb=16,
    ports="22/tcp,11434/http",  # expose SSH and Ollama
    env={
        "OLLAMA_HOST": "0.0.0.0",
        "PYTHONUNBUFFERED": "1"
    },
    support_public_ip=True,
    start_ssh=True
)

pod_id = pod["id"]
print(f"Created pod: {pod_id}")

raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'da2aevsmsa1e23', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'ysysvhe2p7ub', 'machine': {'podHostId': 'da2aevsmsa1e23-64411809'}}}}
Created pod: da2aevsmsa1e23


In [8]:
details = runpod.get_pod(pod_id)
print("DEBUG details:", details)


DEBUG details: {'id': 'da2aevsmsa1e23', 'containerDiskInGb': 200, 'costPerHr': 0.4, 'desiredStatus': 'RUNNING', 'dockerArgs': None, 'dockerId': None, 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'gpuCount': 1, 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'lastStatusChange': 'Rented by User: Thu Sep 25 2025 04:46:34 GMT+0000 (Coordinated Universal Time)', 'machineId': 'ysysvhe2p7ub', 'memoryInGb': 50, 'name': 'lm-eval-pod-test', 'podType': 'RESERVED', 'port': None, 'ports': '22/tcp,11434/http', 'uptimeSeconds': 0, 'vcpuCount': 9, 'volumeInGb': 0, 'volumeMountPath': '/runpod-volume', 'runtime': {'ports': [{'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 11434, 'publicPort': 60831, 'type': 'http'}, {'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 19123, 'publicPort': 60832, 'type': 'http'}, {'ip': '69.30.

## Wait for pod to become RUNNING and get SSH endpoint

In [11]:
# ───────────────────────────────────────────────
# 2. Wait for pod to become RUNNING and get SSH endpoint
# ───────────────────────────────────────────────
import time

ssh_host = None
ssh_port = None

print("Waiting for pod to become RUNNING and for SSH endpoint to appear...")
while True:
    details = runpod.get_pod(pod_id)
    status = details.get("desiredStatus")
    print(f"  Current status: {status}")

    # If the pod is running, check for a public SSH port
    if status == "RUNNING":
        runtime = details.get("runtime")
        if runtime and runtime.get("ports"):
            for p in runtime["ports"]:
                if p["type"] == "tcp" and p["privatePort"] == 22 and p["isIpPublic"]:
                    ssh_host = p["ip"]
                    ssh_port = p["publicPort"]
                    break
            if ssh_host:
                break   # Exit the loop once we have the SSH endpoint

    time.sleep(10)

print(f"Pod is RUNNING with SSH ready at {ssh_host}:{ssh_port}")




Waiting for pod to become RUNNING and for SSH endpoint to appear...
  Current status: RUNNING
Pod is RUNNING with SSH ready at 69.30.85.132:22198


## Connect to the pod

In [5]:
# ───────────────────────────────────────────────
# 3A Connect to the pod
# ───────────────────────────────────────────────
import os
import time
import paramiko
import runpod

# Connect automatically with Paramiko
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())  # auto-accept host fingerprint
ssh.connect(ssh_host, port=ssh_port, username="root", key_filename=ssh_key_path)
print("Connected to pod.")

# Connect with your key (no passphrase)
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
pkey = paramiko.Ed25519Key.from_private_key_file(ssh_key_path)
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(ssh_host, port=ssh_port, username="root", pkey=pkey)
print("SSH connection established.")

Connected to pod.
SSH connection established.


## Provision the pod

In [ ]:
print("Starting provisioning...")

# --- Install prerequisites and Ollama ---
# base_commands = [
#     "apt-get update && apt-get install -y curl git python3-pip lshw",
#     "curl -fsSL https://ollama.com/install.sh | sh",
#     # start Ollama in background and detach so Paramiko doesn't hang
#     "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
#     "sleep 10"   # give the server time to start
# ]

base_commands = [
    # "apt-get update && apt-get install -y curl git python3-pip lshw",
    "apt update && apt install lshw -y",             # optional
    "curl -fsSL https://ollama.com/install.sh | sh", # install Ollama binary
    "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
    "sleep 10",
]

for cmd in base_commands:
    print(f"Running: {cmd}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    print(stdout.read().decode())
    err = stderr.read().decode()
    if err: print("ERROR:", err)

# --- Pull the model ---
model_name = "llama3.2:1b"
print(f"Pulling model {model_name} ...")
stdin, stdout, stderr = ssh.exec_command(f"ollama pull {model_name}")
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Install evaluation harness ---
stdin, stdout, stderr = ssh.exec_command(
    "pip install lm_eval lm_eval[api] ollama==0.3.3"
)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Check until the model appears in 'ollama list' ---
print(f"Waiting for model '{model_name}' to show up in `ollama list`...")
while True:
    stdin, stdout, stderr = ssh.exec_command("ollama list")
    output = stdout.read().decode()
    if model_name.split(":")[0] in output:
        print("Model is ready.")
        break
    print("  Not ready yet, retrying in 10 seconds...")
    time.sleep(10)

print("Provisioning complete.")


Starting provisioning...
Running: apt update && apt install lshw -y
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Reading package lists...
Building dependency tree...
Reading state information...
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists...
Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.

ERROR: 




Running: curl -fsSL https://ollama.com/install.sh | sh

ERROR: >>> Cleaning up old version at /usr/local/lib/ollama
>>> Inst

In [9]:
# Pushing the yaml file to the pod
yaml_content = r'''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  until: []  # No early stopping for thinking models
  max_gen_toks: 10000  # High limit for reasoning traces
  temperature: 0.0  # Deterministic for evaluation

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true'''

remote_file = "/root/sysengbench.yaml"
command = f"cat > {remote_file} <<'EOF'\n{yaml_content}\nEOF"

stdin, stdout, stderr = ssh.exec_command(command)
print(stdout.read().decode(), stderr.read().decode())


## Run the eval on the pod

In [12]:
model = "llama3.2:1b"
base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
include_path = "./"
tasks = "sysengbench"
output_dir = "output/sysengbench/"
log_samples = True
batch_size = "auto"
temperature = 0.0
apply_chat_template = True

# Construct the benchmark command dynamically
log_samples_flag = "--log_samples" if log_samples else ""
apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

lm_eval_cmd = f"""
lm_eval \
  --model local-chat-completions \
  --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
  --include_path {include_path} \
  --tasks {tasks} \
  --output {output_dir} \
  {log_samples_flag} \
  --num_fewshot 0 \
  --batch_size {batch_size} \
  --limit 10 \
  --gen_kwargs temperature={temperature} \
  {apply_template_flag}
"""

stdin, stdout, stderr = ssh.exec_command(lm_eval_cmd)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)


local-chat-completions (model=llama3.2:1b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1), gen_kwargs: (temperature=0.0), limit: 10.0, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |  0.9|±  |   0.1|


ERROR: 2025-09-25:05:30:13 INFO     [__main__:348] Including path: ./
2025-09-25:05:30:16 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-25:05:30:16 INFO     [__main__:446] Selected Tasks: ['sysengbench']
2025-09-25:05:30:16 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-09-25:05:30:16 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will up

## Recover the output files from evals on the pod

In [ ]:
# confirming files are indeed placed in root.
stdin, stdout, stderr = ssh.exec_command(
    "find /root -type f \\( -name 'results_*.json' -o -name 'samples_*.jsonl' \\) 2>/dev/null"
)
matches = stdout.read().decode().strip().splitlines()
print("Found files:", matches)


Found files: ['/root/output/sysengbench/llama3.2__1b/results_2025-09-25T05-30-23.489092.json', '/root/output/sysengbench/llama3.2__1b/samples_sysengbench_2025-09-25T05-30-23.489092.jsonl']


In [15]:
import os
import stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    os.makedirs(local_dir, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir,  entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)  # recurse into subdir
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")


In [ ]:
# permissions check
for path in ["/root", "/root/output", "/root/output/sysengbench"]:
    stdin, stdout, stderr = ssh.exec_command(f"ls -ld {path}")
    print(path, stdout.read().decode(), stderr.read().decode())


/root drwx------ 1 root root 143 Sep 25 05:30 /root
 
/root/output drwxr-xr-x 3 root root 33 Sep 25 05:30 /root/output
 
/root/output/sysengbench drwxr-xr-x 3 root root 34 Sep 25 05:30 /root/output/sysengbench
 


In [21]:
remote_dir = "/root/output"
local_dir  = "C:/Users/rabel/Desktop/dissertation-outputs/runpod_results"

import os, stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    try:
        sftp.chdir(remote_dir)
    except IOError:
        print(f"Remote directory not found: {remote_dir}")
        return
    os.makedirs(local_dir, exist_ok=True)

    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")

# ---- usage ----
sftp = ssh.open_sftp()
sftp_get_dir(sftp, remote_dir, local_dir)
sftp.close()

print(f"All files downloaded to: {local_dir}")


Remote directory not found: /root/output\sysengbench
All files downloaded to: C:/Users/rabel/Desktop/dissertation-outputs/runpod_results


In [23]:
import posixpath  # <-- always uses forward slashes
import os, stat

def sftp_get_dir_fixed(sftp, remote_dir, local_dir, depth=0):
    indent = "  " * depth
    print(f"{indent}Entering remote: {remote_dir}")

    try:
        sftp.chdir(remote_dir)
    except IOError as e:
        print(f"{indent}❌ Cannot access {remote_dir}: {e}")
        return

    os.makedirs(local_dir, exist_ok=True)
    print(f"{indent}Local target: {local_dir}")

    entries = sftp.listdir_attr(remote_dir)
    if not entries:
        print(f"{indent}(empty directory)")
        return

    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)  # ✅ POSIX join
        local_path  = os.path.join(local_dir, entry.filename)      # local join is fine
        print(f"{indent}- {entry.filename} "
              f"{'DIR' if stat.S_ISDIR(entry.st_mode) else 'FILE'}")

        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir_fixed(sftp, remote_path, local_path, depth + 1)
        else:
            try:
                sftp.get(remote_path, local_path)
                print(f"{indent}  ✅ Downloaded to {local_path}")
            except Exception as e:
                print(f"{indent}  ❌ Failed to download {remote_path}: {e}")


In [24]:
remote_dir = "/root/output"
local_dir  = r"C:\Users\rabel\Desktop\dissertation-outputs\runpod_results"

sftp = ssh.open_sftp()
sftp_get_dir_fixed(sftp, remote_dir, local_dir)
sftp.close()


Entering remote: /root/output
Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results
- sysengbench DIR
  Entering remote: /root/output/sysengbench
  Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench
  - llama3.2__1b DIR
    Entering remote: /root/output/sysengbench/llama3.2__1b
    Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b
    - results_2025-09-25T05-30-23.489092.json FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\results_2025-09-25T05-30-23.489092.json
    - samples_sysengbench_2025-09-25T05-30-23.489092.jsonl FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\samples_sysengbench_2025-09-25T05-30-23.489092.jsonl


## Terminate pod

In [25]:
# ───────────────────────────────────────────────
# 6. Terminate pod
# ───────────────────────────────────────────────
runpod.terminate_pod(pod_id)
print(f"Pod {pod_id} terminated.")

ssh.close()

Pod da2aevsmsa1e23 terminated.


## recovering/ reconnecting to a pod

In [3]:
import os, time, runpod

runpod.api_key = os.environ["RUNPOD_API_KEY"]

def reconnect_by_name(pod_name):
    """
    Find an existing RUNNING pod by its name and return its id, public IP and public SSH port.
    """
    pods = runpod.get_pods()
    for p in pods:
        if p["name"] == pod_name and p["desiredStatus"] == "RUNNING":
            details = runpod.get_pod(p["id"])
            runtime = details.get("runtime", {})
            if runtime and runtime.get("ports"):
                for port in runtime["ports"]:
                    if port["type"] == "tcp" and port["privatePort"] == 22 and port["isIpPublic"]:
                        return {
                            "id": details["id"],
                            "host": port["ip"],
                            "port": port["publicPort"]
                        }
    raise RuntimeError(f"No running pod named '{pod_name}' found.")

# Example usage:
pod_name = "lm-eval-pod-test"   # <-- the name you used in create_pod
conn = reconnect_by_name(pod_name)
pod_id, ssh_host, ssh_port = conn["id"], conn["host"], conn["port"]
print(f"Reconnected to {pod_name} -> {ssh_host}:{ssh_port}")


Reconnected to lm-eval-pod-test -> 69.30.85.132:22198


# Dashboard Attempt

In [ ]:
import os
import time
import stat
import runpod
import paramiko
import pandas as pd
from datetime import datetime
from pathlib import Path

# ───────────────────────────────────────────────
# 1. USER CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]
runpod.api_key = RUNPOD_API_KEY

# List of models to evaluate
model_list = [
    # e.g. "llama3.2:1b",
]

# Define evaluation tasks and their output directories
task_configs = [
    ("sysengbench",   "output/sysengbench/"),
    ("sysengbench-a", "output/sysengbench-a/"),
]

IMAGE_NAME   = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
GPU_TYPE     = "NVIDIA A40"     # GPU type for all pods
NUM_CONCURRENT = 1              # number of concurrent requests per lm_eval run
LOCAL_RESULTS_BASE = Path("results")
LOCAL_RESULTS_BASE.mkdir(exist_ok=True)

# ───────────────────────────────────────────────
# 2. DASHBOARD SETUP
# ───────────────────────────────────────────────
dashboard_columns = [
    "Model", "Task", "Pod ID", "GPU Type",
    "Start Time", "Stop Time", "Status"
]
dashboard_df = pd.DataFrame(columns=dashboard_columns)

def add_dashboard_entry(model, task, pod_id, gpu_type):
    global dashboard_df
    dashboard_df = pd.concat([
        dashboard_df,
        pd.DataFrame([{
            "Model": model,
            "Task": task,
            "Pod ID": pod_id,
            "GPU Type": gpu_type,
            "Start Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Stop Time": None,
            "Status": "to do"
        }])
    ], ignore_index=True)

def update_dashboard_status(pod_id, task, status):
    global dashboard_df
    idx = dashboard_df[(dashboard_df["Pod ID"] == pod_id) &
                       (dashboard_df["Task"] == task)].index
    if not idx.empty:
        if status in ("done", "crashed"):
            dashboard_df.loc[idx, "Stop Time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        dashboard_df.loc[idx, "Status"] = status

def show_dashboard():
    """Display the live dashboard in a Jupyter notebook."""
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Pod Run Dashboard", dashboard_df)
    return dashboard_df

# ───────────────────────────────────────────────
# 3. POD UTILITIES
# ───────────────────────────────────────────────
def wait_for_pod_ready(pod_id):
    ssh_host = ssh_port = None
    print("Waiting for pod to be RUNNING & SSH ready...")
    while True:
        details = runpod.get_pod(pod_id)
        if details.get("desiredStatus") == "RUNNING":
            for p in details.get("runtime", {}).get("ports", []):
                if p["type"] == "tcp" and p["privatePort"] == 22 and p["isIpPublic"]:
                    return p["ip"], p["publicPort"]
        time.sleep(10)

def sftp_get_dir_fixed(sftp, remote_dir, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir_fixed(sftp, remote_path, local_path)
        else:
            sftp.get(remote_path, local_path)

def provision_pod(ssh, model):
    """Install Ollama, pull model, install lm_eval inside the pod."""
    cmds = [
        "apt update && apt install -y lshw",
        "curl -fsSL https://ollama.com/install.sh | sh",
        "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
        "sleep 10",
        f"ollama pull {model}",
        "pip install lm_eval lm_eval[api] ollama==0.3.3"
    ]
    for c in cmds:
        print("Running:", c)
        stdin, stdout, stderr = ssh.exec_command(c)
        stdout.channel.recv_exit_status()
        err = stderr.read().decode()
        if err: print("ERROR:", err)

# ───────────────────────────────────────────────
# 4. POD RUNNER
# ───────────────────────────────────────────────
def run_model_in_pod(model, run_all_tasks=True, single_task=None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pod_name  = f"lm-eval-{model.replace(':','-')}-{timestamp}"
    pod = runpod.create_pod(
        name=pod_name,
        image_name=IMAGE_NAME,
        gpu_type_id=GPU_TYPE,
        gpu_count=1,
        container_disk_in_gb=200,
        min_vcpu_count=4,
        min_memory_in_gb=16,
        ports="22/tcp,11434/http",
        env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
        support_public_ip=True,
        start_ssh=True,
    )
    pod_id = pod["id"]
    print(f"\nCreated pod {pod_id} for model {model}")

    # Dashboard rows for each task
    for task_name, _ in task_configs:
        if run_all_tasks or single_task == task_name:
            add_dashboard_entry(model, task_name, pod_id, GPU_TYPE)

    try:
        # Wait for pod readiness
        ssh_host, ssh_port = wait_for_pod_ready(pod_id)
        key_path = os.path.expanduser("~/.ssh/id_ed25519")
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(ssh_host, port=ssh_port, username="root",
                    pkey=paramiko.Ed25519Key.from_private_key_file(key_path))
        print("SSH connected.")
        for task_name, _ in task_configs:
            if run_all_tasks or single_task == task_name:
                update_dashboard_status(pod_id, task_name, "started")

        # Provision environment and model
        provision_pod(ssh, model)

        # Run evaluations
        for task_name, output_dir in task_configs:
            if not run_all_tasks and single_task != task_name:
                continue
            update_dashboard_status(pod_id, task_name, "in progress")
            lm_eval_cmd = f"""
            lm_eval \
              --model local-chat-completions \
              --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent={NUM_CONCURRENT} \
              --include_path ./ \
              --tasks {task_name} \
              --output {output_dir} \
              --log_samples \
              --num_fewshot 0 \
              --batch_size auto \
              --gen_kwargs temperature=0.0 \
              --apply_chat_template
            """
            stdin, stdout, stderr = ssh.exec_command(lm_eval_cmd)
            stdout.channel.recv_exit_status()
            err = stderr.read().decode()
            if err: print("ERROR:", err)
            update_dashboard_status(pod_id, task_name, "done")

        # Download results
        sftp = ssh.open_sftp()
        sftp_get_dir_fixed(sftp, "/root/output",
                           LOCAL_RESULTS_BASE / model.replace(":", "_"))
        sftp.close()

    except Exception as e:
        print(f"ERROR while running model {model}: {e}")
        for task_name, _ in task_configs:
            if run_all_tasks or single_task == task_name:
                update_dashboard_status(pod_id, task_name, "crashed")

    finally:
        runpod.terminate_pod(pod_id)
        print(f"Pod {pod_id} terminated.\n")
        ssh.close()

# ───────────────────────────────────────────────
# 5. MAIN LOOP
# ───────────────────────────────────────────────
for model in model_list:
    run_model_in_pod(model, run_all_tasks=True)
    # Or to run a single task:
    # run_model_in_pod(model, run_all_tasks=False, single_task="sysengbench")

# ───────────────────────────────────────────────
# 6. View Dashboard
# ───────────────────────────────────────────────
# In a Jupyter notebook cell, run:
# show_dashboard()


# Checking for files to be present / Taking Inventory

## Task first column, list all

In [10]:
import os
import pandas as pd
from pathlib import Path

def generate_results_dashboard_auto(base_output_dir: str) -> pd.DataFrame:
    """
    Build a completeness dashboard by automatically discovering
    tasks and models inside a base output directory.

    Structure expected:
        base_output_dir/
            task_name/
                model_name/
                    results_*.json
                    samples_*.jsonl

    Returns
    -------
    pd.DataFrame with columns:
        Task | Model | Results File Found | Samples File Found | Status
    """
    base_path = Path(base_output_dir)
    dashboard_rows = []

    if not base_path.exists():
        raise FileNotFoundError(f"Base directory does not exist: {base_output_dir}")

    # Loop over tasks (e.g., sysengbench, sysengbench-a)
    for task_dir in base_path.iterdir():
        if not task_dir.is_dir():
            continue
        task_name = task_dir.name

        # Loop over models inside each task
        for model_dir in task_dir.iterdir():
            if not model_dir.is_dir():
                continue
            model_name = model_dir.name

            # Check for required files
            has_results = any(
                f.name.startswith("results_") and f.suffix == ".json"
                for f in model_dir.iterdir()
            )
            has_samples = any(
                f.name.startswith("samples_") and f.suffix == ".jsonl"
                for f in model_dir.iterdir()
            )

            status = "complete" if has_results and has_samples else "missing"

            dashboard_rows.append({
                "Task": task_name,
                "Model": model_name,
                "Results File Found": has_results,
                "Samples File Found": has_samples,
                "Status": status
            })

    return pd.DataFrame(dashboard_rows)


In [11]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output (6)\output"

dashboard_df = generate_results_dashboard_auto(base_output_dir)

# Show in Jupyter
display(dashboard_df)

# Optional: save to CSV for later
dashboard_df.to_csv("results_dashboard.csv", index=False)


,Task,Model,Results File Found,Samples File Found,Status
0,sysengbench,gemma3n__e2b,True,True,complete
1,sysengbench,gemma3n__e4b,True,True,complete
2,sysengbench,gemma3__12b,True,True,complete
3,sysengbench,gemma3__1b,True,True,complete
4,sysengbench,gemma3__270m,True,True,complete
5,sysengbench,gemma3__27b,True,True,complete
6,sysengbench,gemma3__4b,True,True,complete
7,sysengbench,gpt-oss__20b,True,True,complete
8,sysengbench,llama3.2__1b,True,True,complete
9,sysengbench,llama3.2__3b,True,True,complete


## Model first column, list all

In [12]:
import pandas as pd
from pathlib import Path

def generate_model_task_dashboard(base_output_dir: str) -> pd.DataFrame:
    """
    Scan a base output directory with structure:
        base_output_dir/task_name/model_name/...
    and build a completeness dashboard listing each model
    and each discovered task, checking for required JSON files.

    Returns
    -------
    pd.DataFrame columns:
        Model | Task | Results File Found | Samples File Found | Status
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all models
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_names.add(model_dir.name)

    dashboard_rows = []

    # Ensure every model appears for every task
    for model in sorted(model_names):
        for task in sorted(task_names):
            model_dir = base_path / task / model
            if model_dir.exists() and model_dir.is_dir():
                has_results = any(
                    f.name.startswith("results_") and f.suffix == ".json"
                    for f in model_dir.iterdir()
                )
                has_samples = any(
                    f.name.startswith("samples_") and f.suffix == ".jsonl"
                    for f in model_dir.iterdir()
                )
                status = "complete" if has_results and has_samples else "missing"
            else:
                has_results = False
                has_samples = False
                status = "missing"

            dashboard_rows.append({
                "Model": model,
                "Task": task,
                "Results File Found": has_results,
                "Samples File Found": has_samples,
                "Status": status
            })

    return pd.DataFrame(dashboard_rows)


In [ ]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output (6)\output"

dashboard_df = generate_model_task_dashboard(base_output_dir)

# Display nicely in Jupyter
display(dashboard_df)



,Model,Task,Results File Found,Samples File Found,Status
0,gemma3__12b,sysengbench,True,True,complete
1,gemma3__12b,sysengbench-a,True,True,complete
2,gemma3__12b,sysengbench-b,True,True,complete
3,gemma3__12b,sysengbench-c,True,True,complete
4,gemma3__12b,sysengbench-d,True,True,complete
...,...,...,...,...,...
70,qwen3__8b,sysengbench,True,True,complete
71,qwen3__8b,sysengbench-a,True,True,complete
72,qwen3__8b,sysengbench-b,False,False,missing
73,qwen3__8b,sysengbench-c,False,False,missing


## Model first column, column check boxes for each task

In [14]:
import pandas as pd
from pathlib import Path

def generate_model_task_matrix(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix dashboard where rows are models and columns are tasks.
    Each cell is '✅' if both results_*.json and samples_*.jsonl exist,
    otherwise '❌'.

    Structure expected:
        base_output_dir/task_name/model_name/...
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all models
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_names.add(model_dir.name)

    # Initialize matrix with ❌
    matrix = pd.DataFrame("❌", index=sorted(model_names), columns=sorted(task_names))

    # Fill in ✅ where both files are present
    for task in task_names:
        for model in model_names:
            model_dir = base_path / task / model
            if model_dir.exists():
                has_results = any(
                    f.name.startswith("results_") and f.suffix == ".json"
                    for f in model_dir.iterdir()
                )
                has_samples = any(
                    f.name.startswith("samples_") and f.suffix == ".jsonl"
                    for f in model_dir.iterdir()
                )
                if has_results and has_samples:
                    matrix.at[model, task] = "✅"

    matrix.index.name = "Model"
    return matrix


In [ ]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output (6)\output"

matrix_df = generate_model_task_matrix(base_output_dir)

# Display in Jupyter
display(matrix_df)

# Optional: export for sharing
# matrix_df.to_csv("model_task_matrix.csv")
# matrix_df.to_html("model_task_matrix.html")

# ✅ means both results_*.json and samples_*.jsonl were found.
# ❌ means one or both files are missing.
# All discovered tasks become columns automatically.


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d
Model,,,,,
gemma3__12b,✅,✅,✅,✅,✅
gemma3__1b,✅,✅,✅,✅,✅
gemma3__270m,✅,✅,✅,✅,✅
gemma3__27b,✅,✅,✅,✅,✅
gemma3__4b,✅,✅,✅,✅,✅
gemma3n__e2b,✅,✅,✅,✅,✅
gemma3n__e4b,✅,✅,✅,✅,✅
gpt-oss__20b,✅,❌,❌,❌,❌
llama3.2__1b,✅,❌,❌,❌,❌


## Extracting the missing models and tasks for the model_list = []

In [17]:
def extract_missing_model_tasks(matrix_df: pd.DataFrame) -> dict:
    """
    Given a model × task matrix of '✅'/'❌',
    return a dict mapping each model to a list of tasks that are missing.
    Only models with at least one missing task are included.
    
    Example return:
    {
        "gemma3_1b": ["sysengbench-a"],
        "llama3.2_8b": ["sysengbench", "sysengbench-a"]
    }
    """
    missing = {}
    for model, row in matrix_df.iterrows():
        missing_tasks = [task for task, value in row.items() if value == "❌"]
        if missing_tasks:
            missing[model] = missing_tasks
    return missing


In [18]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output (6)\output"

# First generate the model×task matrix
matrix_df = generate_model_task_matrix(base_output_dir)

# Now extract all missing tasks per model
missing_dict = extract_missing_model_tasks(matrix_df)

print(missing_dict)


{'gpt-oss__20b': ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'llama3.2__1b': ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'llama3.2__3b': ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'mistral-small3.2__24b': ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'qwen3__0.6b': ['sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'qwen3__1.7b': ['sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'qwen3__4b': ['sysengbench-b', 'sysengbench-c', 'sysengbench-d'], 'qwen3__8b': ['sysengbench-b', 'sysengbench-c', 'sysengbench-d']}


## Adding the number of run counts to the table

In [20]:
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_counts(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix dashboard where rows are models and columns are tasks.
    Each cell is '✅ (n)' if n results_*.json files exist and at least
    one samples_*.jsonl exists, or '❌ (0)' if missing.
    This makes it easy to spot duplicates (n>1).
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and models
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_names.add(model_dir.name)

    # Initialize matrix with ❌ (0)
    matrix = pd.DataFrame("❌ (0)", index=sorted(model_names), columns=sorted(task_names))

    # Fill in ✅ (n) where applicable
    for task in task_names:
        for model in model_names:
            model_dir = base_path / task / model
            if model_dir.exists():
                result_files = [f for f in model_dir.iterdir() if f.name.startswith("results_") and f.suffix == ".json"]
                samples_exist = any(f.name.startswith("samples_") and f.suffix == ".jsonl"
                                    for f in model_dir.iterdir())
                n_results = len(result_files)
                if n_results > 0 and samples_exist:
                    matrix.at[model, task] = f"✅ ({n_results})"
                else:
                    matrix.at[model, task] = f"❌ ({n_results})"

    matrix.index.name = "Model"
    return matrix


In [24]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output"

matrix_df = generate_model_task_matrix_with_counts(base_output_dir)

# Display in Jupyter
display(matrix_df)

# Optional export
matrix_df.to_csv("model_task_matrix_with_counts.csv")


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d
Model,,,,,
devstral__24b,✅ (1),❌ (0),❌ (0),❌ (0),❌ (0)
gemma3__12b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3__1b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3__270m,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3__27b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3__4b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3n__e2b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gemma3n__e4b,✅ (2),✅ (1),✅ (1),✅ (1),✅ (1)
gpt-oss__20b,✅ (3),❌ (0),❌ (0),❌ (0),❌ (0)


## Adding the highest score from all of the results for that model+task combo

In [25]:
import json
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_scores(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix dashboard where rows are models and columns are tasks.
    Each cell shows:
        '✅ (n) – max:score' if n result files exist, at least one samples file exists,
        and displays the highest exact_match score found.
        '❌ (n) – max:score' otherwise (score may be 0 if no results).

    Directory structure expected:
        base_output_dir/task_name/model_name/results_*.json
                                       /samples_*.jsonl
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all models
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_names.add(model_dir.name)

    # Initialize matrix with ❌ (0 – max:0.0)
    matrix = pd.DataFrame("❌ (0 – max:0.0)", index=sorted(model_names), columns=sorted(task_names))

    # Fill matrix with counts and highest score
    for task in task_names:
        for model in model_names:
            model_dir = base_path / task / model
            if model_dir.exists():
                result_files = [f for f in model_dir.iterdir()
                                if f.name.startswith("results_") and f.suffix == ".json"]
                n_results = len(result_files)

                # Find the highest score among all result files
                max_score = 0.0
                for rf in result_files:
                    try:
                        with open(rf, "r") as fh:
                            data = json.load(fh)
                        # Get the score safely
                        if "results" in data and task in data["results"]:
                            # Look for exact_match or similar key
                            task_data = data["results"][task]
                            # Try exact_match or first numeric value
                            score = task_data.get("exact_match,strict-match") \
                                    or task_data.get("exact_match") \
                                    or max((v for v in task_data.values() if isinstance(v,(int,float))), default=0.0)
                            if score and score > max_score:
                                max_score = float(score)
                    except Exception as e:
                        print(f"Warning: could not parse {rf}: {e}")

                samples_exist = any(
                    f.name.startswith("samples_") and f.suffix == ".jsonl"
                    for f in model_dir.iterdir()
                )

                if n_results > 0 and samples_exist:
                    matrix.at[model, task] = f"✅ ({n_results}) – max:{max_score:.3f}"
                else:
                    matrix.at[model, task] = f"❌ ({n_results}) – max:{max_score:.3f}"

    matrix.index.name = "Model"
    return matrix


In [27]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output"

matrix_df = generate_model_task_matrix_with_scores(base_output_dir)

# Show in Jupyter
display(matrix_df)

# Optional export
# matrix_df.to_csv("model_task_matrix_with_scores.csv")


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d
Model,,,,,
deepseek-r1__14b,❌ (0 – max:0.0),✅ (1) – max:0.000,✅ (1) – max:0.000,✅ (1) – max:0.000,❌ (0 – max:0.0)
deepseek-r1__32b,❌ (0 – max:0.0),✅ (1) – max:0.000,✅ (1) – max:0.000,✅ (1) – max:0.000,❌ (0 – max:0.0)
deepseek-r1__70b,❌ (0 – max:0.0),✅ (1) – max:0.000,✅ (1) – max:0.000,✅ (1) – max:0.000,❌ (0 – max:0.0)
deepseek-r1__8b,❌ (0 – max:0.0),✅ (2) – max:0.000,✅ (1) – max:0.000,✅ (1) – max:0.000,❌ (0 – max:0.0)
devstral__24b,✅ (1) – max:0.931,✅ (1) – max:0.898,✅ (1) – max:0.941,✅ (1) – max:0.941,❌ (0 – max:0.0)
gemma3__12b,✅ (2) – max:0.892,✅ (2) – max:0.865,✅ (2) – max:0.894,✅ (2) – max:0.906,✅ (1) – max:0.892
gemma3__1b,✅ (2) – max:0.691,✅ (2) – max:0.748,✅ (2) – max:0.552,✅ (2) – max:0.794,✅ (1) – max:0.753
gemma3__270m,✅ (2) – max:0.136,✅ (2) – max:0.108,✅ (2) – max:0.007,✅ (2) – max:0.317,✅ (1) – max:0.010
gemma3__27b,✅ (2) – max:0.920,✅ (2) – max:0.903,✅ (2) – max:0.918,✅ (2) – max:0.932,✅ (1) – max:0.911


## Adding the number of samples as well 

In [28]:
import json
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_scores_and_samples(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix where each cell shows:
        '✅ (n) – max:score – samples:m'
    if n results exist, at least one matching samples file exists, and the
    highest-scoring run's sample file line count is reported.

    Directory structure expected:
        base_output_dir/task_name/model_name/
            results_<timestamp>.json
            samples_<task>_<timestamp>.jsonl
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover tasks and models
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_names.add(model_dir.name)

    matrix = pd.DataFrame("❌ (0 – max:0.0 – samples:0)", index=sorted(model_names), columns=sorted(task_names))

    for task in task_names:
        for model in model_names:
            model_dir = base_path / task / model
            if not model_dir.exists():
                continue

            result_files = [f for f in model_dir.iterdir()
                            if f.name.startswith("results_") and f.suffix == ".json"]

            if not result_files:
                continue

            best_score = 0.0
            best_result_file = None

            # find highest scoring results file
            for rf in result_files:
                try:
                    with open(rf, "r") as fh:
                        data = json.load(fh)
                    if "results" in data and task in data["results"]:
                        task_data = data["results"][task]
                        score = (task_data.get("exact_match,strict-match")
                                 or task_data.get("exact_match")
                                 or max((v for v in task_data.values() if isinstance(v, (int, float))), default=0.0))
                        if score and score > best_score:
                            best_score = float(score)
                            best_result_file = rf
                except Exception as e:
                    print(f"Warning: could not parse {rf}: {e}")

            n_results = len(result_files)
            samples_exist = False
            max_samples_count = 0

            if best_result_file:
                # Extract the timestamp from best_result_file name
                # Filename format expected: results_<timestamp>.json
                ts = best_result_file.stem.replace("results_", "")
                expected_samples_prefix = f"samples_{task}_{ts}"
                # Find the matching samples file
                for f in model_dir.iterdir():
                    if f.name.startswith(expected_samples_prefix) and f.suffix == ".jsonl":
                        samples_exist = True
                        # count lines (samples)
                        with open(f, "r", encoding="utf-8") as sf:
                            count = sum(1 for _ in sf)
                        max_samples_count = count
                        break  # only need one matching file

            if best_result_file and samples_exist:
                matrix.at[model, task] = f"✅ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"
            else:
                matrix.at[model, task] = f"❌ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"

    matrix.index.name = "Model"
    return matrix


In [29]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output"

matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

# View in Jupyter
display(matrix_df)

# Optional export
# matrix_df.to_csv("model_task_matrix_with_scores_and_samples.csv")


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d
Model,,,,,
deepseek-r1__14b,❌ (0 – max:0.0 – samples:0),❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (0 – max:0.0 – samples:0)
deepseek-r1__32b,❌ (0 – max:0.0 – samples:0),❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (0 – max:0.0 – samples:0)
deepseek-r1__70b,❌ (0 – max:0.0 – samples:0),❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (0 – max:0.0 – samples:0)
deepseek-r1__8b,❌ (0 – max:0.0 – samples:0),❌ (2) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (0 – max:0.0 – samples:0)
devstral__24b,✅ (1) – max:0.931 – samples:1144,✅ (1) – max:0.898 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.941 – samples:1144,❌ (0 – max:0.0 – samples:0)
gemma3__12b,✅ (2) – max:0.892 – samples:1144,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.894 – samples:1144,✅ (2) – max:0.906 – samples:1144,✅ (1) – max:0.892 – samples:1144
gemma3__1b,✅ (2) – max:0.691 – samples:1144,✅ (2) – max:0.748 – samples:1144,✅ (2) – max:0.552 – samples:1144,✅ (2) – max:0.794 – samples:1144,✅ (1) – max:0.753 – samples:1144
gemma3__270m,✅ (2) – max:0.136 – samples:1144,✅ (2) – max:0.108 – samples:1144,✅ (2) – max:0.007 – samples:1144,✅ (2) – max:0.317 – samples:1144,✅ (1) – max:0.010 – samples:1144
gemma3__27b,✅ (2) – max:0.920 – samples:1144,✅ (2) – max:0.903 – samples:1144,✅ (2) – max:0.918 – samples:1144,✅ (2) – max:0.932 – samples:1144,✅ (1) – max:0.911 – samples:1144
